# Notebook 07 — AI Deployment (FastAPI + Streamlit)

Kiến trúc triển khai: **User → Streamlit → FastAPI → Pydantic validation → shared FeatureBuilder → fitted preprocessing → final model → popularity prediction**.

User không nhập engineered features. Notebook này kiểm tra artifact, feature order, `/health`, valid `/predict`, invalid input và parity giữa API với pipeline trực tiếp.

In [1]:
from pathlib import Path
import importlib.util
import json
import sys

import joblib
import numpy as np
import pandas as pd
from fastapi.testclient import TestClient

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for candidate in Path.cwd().resolve().parents:
        if (candidate / "src").exists() and (candidate / "5.DATA").exists():
            ROOT = candidate
            break
sys.path.insert(0, str(ROOT))

from src.features import (
    EXPECTED_ENGINEERED_FEATURES,
    MODEL_FEATURES,
    RAW_INPUT_FEATURES,
)

ARTIFACT_DIR = ROOT / "4.MODELS" / "hitradar_popularity"
PIPELINE_PATH = ARTIFACT_DIR / "popularity_pipeline.joblib"
FEATURE_COLUMNS_PATH = ARTIFACT_DIR / "feature_columns.json"
API_PATH = ROOT / "5.UNG_DUNG" / "5.1.backend_api" / "api.py"
STREAMLIT_PATH = ROOT / "5.UNG_DUNG" / "5.2.frontend" / "streamlit_app.py"

assert PIPELINE_PATH.exists(), "Chưa có final pipeline từ Notebook 06."
assert API_PATH.exists() and API_PATH.stat().st_size > 0
assert STREAMLIT_PATH.exists() and STREAMLIT_PATH.stat().st_size > 0
print(f"Model artifact: {PIPELINE_PATH}")
print(f"FastAPI source: {API_PATH}")
print(f"Streamlit source: {STREAMLIT_PATH}")

D:\Hitradar\hitradar-main\scratch\runtime_packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Model artifact: D:\Hitradar\hitradar-main\4.MODELS\hitradar_popularity\popularity_pipeline.joblib
FastAPI source: D:\Hitradar\hitradar-main\5.UNG_DUNG\5.1.backend_api\api.py
Streamlit source: D:\Hitradar\hitradar-main\5.UNG_DUNG\5.2.frontend\streamlit_app.py


## 1. Model load và train/inference contract

In [2]:
pipeline = joblib.load(PIPELINE_PATH)
feature_contract = json.loads(FEATURE_COLUMNS_PATH.read_text(encoding="utf-8"))

assert feature_contract["raw_input_features"] == RAW_INPUT_FEATURES
assert feature_contract["model_features"] == MODEL_FEATURES
assert feature_contract["engineered_features"] == EXPECTED_ENGINEERED_FEATURES
assert pipeline.named_steps["features"].get_feature_names_out().tolist() == MODEL_FEATURES

print(f"Raw fields: {len(RAW_INPUT_FEATURES)}")
print(f"Engineered fields generated server-side: {len(EXPECTED_ENGINEERED_FEATURES)}")
print(f"Model features before encoding: {len(MODEL_FEATURES)}")
print(f"Encoded matrix columns: {len(feature_contract['transformed_feature_names'])}")

Raw fields: 17
Engineered fields generated server-side: 13
Model features before encoding: 31
Encoded matrix columns: 76


## 2. RAW INPUT → FEATURE ENGINEERING → MODEL → PREDICTION

In [3]:
sample_raw = {
    "duration_min": 3.55,
    "explicit": False,
    "release_year": 2020,
    "release_month": 7.0,
    "release_precision": "day",
    "danceability": 0.72,
    "energy": 0.78,
    "key": 5,
    "loudness": -6.5,
    "mode": 1,
    "speechiness": 0.08,
    "acousticness": 0.18,
    "instrumentalness": 0.02,
    "liveness": 0.14,
    "valence": 0.64,
    "tempo": 124.0,
    "time_signature": 4.0,
}
raw_frame = pd.DataFrame([sample_raw])[RAW_INPUT_FEATURES]
engineered_frame = pipeline.named_steps["features"].transform(raw_frame)
missing_features = [feature for feature in MODEL_FEATURES if feature not in engineered_frame.columns]
assert not missing_features
assert np.isfinite(engineered_frame.select_dtypes(include=np.number)).all().all()

direct_prediction = float(np.clip(pipeline.predict(raw_frame)[0], 0, 100))
print(f"RAW shape: {raw_frame.shape}")
print(f"Feature shape before encoding: {engineered_frame.shape}")
print(f"Direct prediction: {direct_prediction:.4f}")
display(engineered_frame[EXPECTED_ENGINEERED_FEATURES])

RAW shape: (1, 17)
Feature shape before encoding: (1, 31)
Direct prediction: 40.8421


,key_sin,key_cos,dance_energy,positive_energy,acoustic_energy_balance,dance_valence,acoustic_instrumental,tempo_energy,energy_vs_period_avg,dance_vs_period_avg,mood_quadrant,duration_category,tempo_category
0,0.5,-0.866025,0.5616,0.4992,0.1875,0.4608,0.0036,96.72,0.243319,0.161829,high_energy_positive,standard,fast


## 3. FastAPI /health, /predict và invalid input

In [4]:
spec = importlib.util.spec_from_file_location("hitradar_api", API_PATH)
api_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(api_module)
client = TestClient(api_module.app)

health_response = client.get("/health")
assert health_response.status_code == 200
health = health_response.json()
assert health["status"] == "ready" and health["model_loaded"] is True

predict_response = client.post("/predict", json=sample_raw)
assert predict_response.status_code == 200, predict_response.text
api_prediction = predict_response.json()
assert abs(api_prediction["predicted_popularity"] - direct_prediction) < 1e-3
assert api_prediction["engineered_feature_count"] == len(EXPECTED_ENGINEERED_FEATURES)
assert api_prediction["feature_count"] == len(MODEL_FEATURES)

invalid_payload = {**sample_raw, "energy": 1.5}
invalid_response = client.post("/predict", json=invalid_payload)
assert invalid_response.status_code == 422

engineered_input_attack = {**sample_raw, "key_sin": 0.5}
extra_field_response = client.post("/predict", json=engineered_input_attack)
assert extra_field_response.status_code == 422

test_results = pd.DataFrame([
    {"Test": "Model load", "Status": "PASS"},
    {"Test": "GET /health", "Status": "PASS"},
    {"Test": "POST /predict valid raw input", "Status": "PASS"},
    {"Test": "Invalid range rejected", "Status": "PASS"},
    {"Test": "Engineered input rejected", "Status": "PASS"},
    {"Test": "Direct/API prediction parity", "Status": "PASS"},
    {"Test": "Feature names and order", "Status": "PASS"},
    {"Test": "Streamlit source present", "Status": "PASS"},
])
display(test_results)
print(api_prediction)

,Test,Status
0,Model load,PASS
1,GET /health,PASS
2,POST /predict valid raw input,PASS
3,Invalid range rejected,PASS
4,Engineered input rejected,PASS
5,Direct/API prediction parity,PASS
6,Feature names and order,PASS
7,Streamlit source present,PASS


{'predicted_popularity': 40.8421, 'popularity_tier': 'emerging', 'model_name': 'XGBoost', 'engineered_feature_count': 13, 'feature_count': 31}


## 4. Save deployment smoke-test evidence

In [5]:
VALIDATION_DIR = ROOT / "5.UNG_DUNG" / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
TEST_RESULT_PATH = VALIDATION_DIR / "hard_requirement_deployment_smoke_test.json"

payload = {
    "all_pass": bool(test_results["Status"].eq("PASS").all()),
    "tests": test_results.to_dict(orient="records"),
    "health": health,
    "valid_prediction": api_prediction,
    "invalid_status_code": invalid_response.status_code,
    "extra_engineered_field_status_code": extra_field_response.status_code,
    "raw_input_features": RAW_INPUT_FEATURES,
    "model_features": MODEL_FEATURES,
}
TEST_RESULT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved smoke-test evidence: {TEST_RESULT_PATH}")

Saved smoke-test evidence: D:\Hitradar\hitradar-main\5.UNG_DUNG\validation\hard_requirement_deployment_smoke_test.json


## 5. Insight, cách chạy và kết luận

**Finding:** model load, health, valid prediction, invalid-input rejection, feature order và direct/API parity đều PASS.  
**Interpretation:** feature engineering ở inference không phải bản copy khác; API load đúng pipeline đã fit tại Notebook 06.  
**Impact:** user chỉ nhập raw audio/time fields; 13 engineered features và encoding được xử lý server-side, loại bỏ mismatch kiểu “train 31 features nhưng API gửi 11”.  
**Limitations:** FastAPI/Streamlit smoke test trong notebook xác nhận logic và contract; kiểm thử tải, authentication và production observability vẫn nằm ngoài phạm vi.

Chạy ứng dụng từ project root:

```powershell
uvicorn api:app --app-dir "5.UNG_DUNG/5.1.backend_api" --host 127.0.0.1 --port 8000
streamlit run "5.UNG_DUNG/5.2.frontend/streamlit_app.py"
```